# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **Clear Agent Instructions** — Crafting precise, role-defining prompts that guide agent behavior
2. **Structured Output with Pydantic Models** — Ensuring agents return predictable, validated data
3. **Single Responsibility Agents** — Designing focused agents that each do one thing well

We'll apply each pattern to a **travel destination recommender** scenario, progressively building a system that can suggest destinations, check availability, and handle logistics.

## Setup

In [1]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import Agent, tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

project_endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")

if not project_endpoint:
    raise ValueError("Please set AZURE_AI_PROJECT_ENDPOINT in your environment.")

c:\Work\agentsdemo\ai-agents-for-beginners\venv\Lib\site-packages\agent_framework\_skills.py:121: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Work\agentsdemo\ai-agents-for-beginners\venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


## Pattern 1: Clear Agent Instructions

The most impactful pattern is also the simplest: writing clear, detailed instructions for your agent.

Good instructions define:
- **Who** the agent is (persona and tone)
- **What** it should do (step-by-step responsibilities)
- **How** it should behave (constraints and style)

Below, we create a travel concierge agent with explicit instructions that shape every response it produces.

In [3]:
client = FoundryChatClient(
    project_endpoint=project_endpoint,
    model=model,
    credential=AzureCliCredential(),
)

agent = Agent(
    client=client,
    name="TravelConcierge",
    instructions=(
        """You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel."""
    )
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

That sounds like a wonderful getaway! Thank you for sharing your preferences. To design the perfect trip, I’ll propose destinations steeped in history and culture, while also offering excellent cuisine—all within your $2,500 budget. Let me ask a couple of quick clarifications first:  

1. Do you already have a preferred region in mind (e.g., Europe, Asia, or the Americas)?
2. Are you open to international destinations requiring a visa, or would you prefer something with simplified entry requirements?

Once I have these details, I’ll craft an ideal week-long itinerary for you!


## Pattern 2: Structured Output with Pydantic Models

Free-form text is useful for conversation, but downstream systems need structured data.
By pairing **Pydantic models** with a **tool function**, we can:

- Define an exact schema for the agent's output
- Validate responses automatically
- Integrate agent results into application logic reliably

We also introduce a tool that returns destination details so the agent grounds its recommendations in real data.

In [4]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = Agent(
    client=client,
    tools=[get_destination_details],
    name="StructuredTravelExpert",
    instructions=(
       "You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool."
    ))


response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

Here are three destinations ideal for a culture-loving traveler with a $2,500 budget:

1. **Kyoto, Japan**: A treasure trove of traditional culture with countless temples, shrines, and geishas in historic districts like Gion. Enjoy serene gardens, tea ceremonies, and the beautiful Arashiyama Bamboo Grove.

2. **Florence, Italy**: The heart of the Renaissance movement, Florence is home to famous art, architecture, and museums like the Uffizi Gallery. Stroll along the Ponte Vecchio and enjoy Tuscan cuisine.

3. **Mexico City, Mexico**: A vibrant city offering a unique blend of ancient and contemporary culture, visit sites like the ancient Teotihuacan pyramids, Frida Kahlo’s Casa Azul, and bustling artisan markets.

Let me know if you’d like tailored suggestions or more details!


## Pattern 3: Single Responsibility Agents

Complex tasks benefit from splitting work across multiple focused agents, each with a single responsibility:

- A **Destination Expert** that knows about places and availability
- A **Logistics Planner** that handles flights, hotels, and itineraries

This mirrors the software engineering principle of *separation of concerns* — each agent is easier to test, maintain, and improve independently.

In [6]:
destination_agent = Agent(
    client=client,
    tools=[get_destination_details],
    name="DestinationExpert",
    instructions=(
       """You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that."""
)
)

logistics_agent = Agent(
    client=client,
    tools=[get_destination_details],
    name="LogisticsPlanner",
    instructions=(
       """You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that."""
)
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    "Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

=== Destination Expert ===
Here’s a shortlist of destinations ideal for a cultural and food-oriented getaway under $2,500:

### 1. **Mexico City, Mexico**
   - **Pros**: Known for its rich culinary scene (tacos, mole, street markets) and vibrant cultural landmarks like ancient Aztec ruins, museums, and colorful neighborhoods.
   - **Cons**: Can be crowded, and air quality is sometimes a concern.

### 2. **Bangkok, Thailand**
   - **Pros**: A hub for street food, bustling markets, and stunning temples. Affordable activities make it budget-friendly.
   - **Cons**: Traffic and humidity can be overwhelming.

### 3. **Rome, Italy**
   - **Pros**: Home to ancient ruins, world-famous art, and authentic Italian cuisine. Ideal for historical and culinary exploration.
   - **Cons**: Slightly pricier than the other options and busy tourist spots.

Each of these destinations offers exceptional food and culture within your budget. Let me know if one stands out!

=== Logistics Planner ===
It seems I

## Summary

In this lesson we applied three agentic design patterns to a travel recommender scenario:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | Define persona, responsibilities, and constraints up front | Consistent, on-brand agent behavior |
| **Structured Output** | Use Pydantic models as the response format | Validated, machine-readable results |
| **Single Responsibility** | Give each agent one focused job | Easier to test, maintain, and compose |

These patterns compose naturally — you can combine clear instructions with structured output inside a single-responsibility agent to build robust, production-ready systems.